<a href="https://colab.research.google.com/github/viraj97-sl/ai-ml-ds-learning-hub/blob/master/01_Data_Scientist/intermediate/05_ensemble_methods.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Ensemble Methods

Ensemble methods combine multiple models to produce a stronger predictor. They win Kaggle competitions and dominate production tabular ML.

**Topics:** Bagging, Boosting (AdaBoost, XGBoost, LightGBM, CatBoost), Stacking, Voting, feature importance

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (RandomForestClassifier, ExtraTreesClassifier, AdaBoostClassifier,
                               GradientBoostingClassifier, VotingClassifier, StackingClassifier,
                               BaggingClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import warnings; warnings.filterwarnings('ignore')

np.random.seed(42)

X, y = make_classification(n_samples=2000, n_features=20, n_informative=12, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f'Dataset: {X_train.shape[0]} train, {X_test.shape[0]} test')

## 1. Bagging — Bootstrap Aggregating

In [ ]:
# Bagging: train many diverse models on bootstrap samples, average predictions
# Key: diversity from randomness → reduces variance without increasing bias

# Manual bagging to understand the concept
n_estimators = 100
base_accuracies = []
ensemble_preds = np.zeros(len(X_test))

for i in range(n_estimators):
    # Bootstrap sample
    idx = np.random.choice(len(X_train), len(X_train), replace=True)
    X_boot, y_boot = X_train[idx], y_train[idx]
    
    # Train base model
    base = DecisionTreeClassifier(max_depth=None, random_state=i)
    base.fit(X_boot, y_boot)
    
    pred = base.predict(X_test)
    base_accuracies.append((pred == y_test).mean())
    ensemble_preds += pred

# Majority vote
ensemble_vote = (ensemble_preds > n_estimators / 2).astype(int)
ensemble_acc = (ensemble_vote == y_test).mean()

print(f'Individual tree accuracy: {np.mean(base_accuracies):.4f} ± {np.std(base_accuracies):.4f}')
print(f'Ensemble (manual bagging): {ensemble_acc:.4f}')

# Compare single tree vs Random Forest (sophisticated bagging with feature subsampling)
single_tree = cross_val_score(DecisionTreeClassifier(), X_train, y_train, cv=5).mean()
random_forest = cross_val_score(RandomForestClassifier(n_estimators=100, n_jobs=-1), X_train, y_train, cv=5).mean()
extra_trees = cross_val_score(ExtraTreesClassifier(n_estimators=100, n_jobs=-1), X_train, y_train, cv=5).mean()

print(f'\nSingle Decision Tree (CV): {single_tree:.4f}')
print(f'Random Forest      (CV): {random_forest:.4f}  (+{random_forest-single_tree:.4f})')
print(f'Extra Trees        (CV): {extra_trees:.4f}  (+{extra_trees-single_tree:.4f})')

## 2. Boosting

In [ ]:
# Boosting: sequentially add models, each correcting errors of the previous
# Key: models are not independent — each learns from predecessor's mistakes

# Show how boosting iteratively improves
n_stages = [1, 5, 10, 25, 50, 100, 200]
gbm = GradientBoostingClassifier(n_estimators=200, learning_rate=0.1, max_depth=3, random_state=42)
gbm.fit(X_train, y_train)

# staged_predict gives predictions at each stage
train_scores = []
test_scores = []
for n, pred in enumerate(gbm.staged_predict(X_train), 1):
    train_scores.append((pred == y_train).mean())
for n, pred in enumerate(gbm.staged_predict(X_test), 1):
    test_scores.append((pred == y_test).mean())

plt.figure(figsize=(10, 5))
plt.plot(train_scores, 'b-', lw=2, label='Train accuracy')
plt.plot(test_scores, 'r-', lw=2, label='Test accuracy')
plt.xlabel('Number of boosting stages'); plt.ylabel('Accuracy')
plt.title('Gradient Boosting: Accuracy vs Number of Estimators')
plt.legend(); plt.grid(True, alpha=0.3); plt.tight_layout(); plt.show()

# Compare boosting methods
boosting_models = {
    'AdaBoost':          AdaBoostClassifier(n_estimators=100, random_state=42),
    'GradientBoosting':  GradientBoostingClassifier(n_estimators=100, random_state=42),
}

# Try XGBoost if available
try:
    from xgboost import XGBClassifier
    boosting_models['XGBoost'] = XGBClassifier(n_estimators=100, random_state=42, eval_metric='logloss')
except ImportError:
    print('Note: Install xgboost for best performance: pip install xgboost')

try:
    from lightgbm import LGBMClassifier
    boosting_models['LightGBM'] = LGBMClassifier(n_estimators=100, random_state=42, verbose=-1)
except ImportError:
    print('Note: Install lightgbm: pip install lightgbm')

print('\nBoosting comparison (5-fold CV):')
for name, model in boosting_models.items():
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='roc_auc')
    print(f'  {name:<25} AUC={scores.mean():.4f} ± {scores.std():.4f}')

## 3. Feature Importance

In [ ]:
# Random Forest feature importance
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

feature_names = [f'feature_{i}' for i in range(X_train.shape[1])]
importances = pd.Series(rf.feature_importances_, index=feature_names).sort_values(ascending=False)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar chart
importances.head(10).plot(kind='bar', ax=axes[0], color='steelblue', alpha=0.8)
axes[0].set_title('Top 10 Feature Importances (Random Forest)')
axes[0].set_ylabel('Importance'); axes[0].tick_params(axis='x', rotation=45)

# Permutation importance (more reliable)
from sklearn.inspection import permutation_importance
perm_imp = permutation_importance(rf, X_test, y_test, n_repeats=10, random_state=42)
perm_series = pd.Series(perm_imp.importances_mean, index=feature_names).sort_values(ascending=False)
perm_series.head(10).plot(kind='bar', ax=axes[1], color='darkorange', alpha=0.8)
axes[1].set_title('Top 10 Permutation Importances (more reliable)')
axes[1].set_ylabel('Importance'); axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout(); plt.show()
print('Note: Permutation importance measures ACTUAL impact on predictions')
print('     Built-in importance can be biased toward high-cardinality features')

## 4. Stacking

In [ ]:
# Stacking: train base models, use their predictions as features for a meta-model
base_models = [
    ('rf', RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)),
    ('gbm', GradientBoostingClassifier(n_estimators=50, random_state=42)),
    ('lr', Pipeline([('s', StandardScaler()), ('clf', LogisticRegression(max_iter=1000))])),
]
meta_model = LogisticRegression(max_iter=1000)

stacking = StackingClassifier(
    estimators=base_models,
    final_estimator=meta_model,
    cv=5,
    stack_method='predict_proba'
)

# Compare all ensemble strategies
results = {}
for name, model in [
    ('Single Decision Tree', DecisionTreeClassifier(random_state=42)),
    ('Random Forest', RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)),
    ('Gradient Boosting', GradientBoostingClassifier(n_estimators=100, random_state=42)),
    ('Stacking', stacking),
]:
    scores = cross_val_score(model, X_train, y_train, cv=5, scoring='roc_auc')
    results[name] = scores.mean()
    print(f'{name:<25} AUC={scores.mean():.4f}')

plt.figure(figsize=(8, 4))
plt.bar(results.keys(), results.values(), color='steelblue', alpha=0.8)
plt.ylim(0.85, 1.0); plt.xticks(rotation=20)
plt.ylabel('AUC-ROC (5-fold CV)')
plt.title('Ensemble Strategy Comparison')
plt.tight_layout(); plt.show()